# 02 · Biological images are arrays

    **Audience:** first-year biology students  
    **Time:** about 70 minutes  
    **Prerequisites:** Tutorial 01 or equivalent Python basics  
    **Data:** generated locally; no downloads or personal/clinical data.

    ## By the end, you can

- explain a grayscale image as a 2D NumPy array
- crop, normalize and visualize an image
- connect pixel values to intensity measurements

    **How to work:** Run cells from top to bottom (Shift+Enter). Inspect every figure, try the **Your turn** prompts, then compare with the supplied self-checks. A GPU is not necessary unless noted. If you restart the runtime, rerun all cells.

## 1 · What does a fluorescence image measure?
Microscope detectors produce numbers. A 2D grayscale image is a table of intensities with shape `(height, width)`. Here we **simulate** a fluorescence microscopy frame with three bright nuclear-shaped objects, camera noise and uneven illumination. Nothing is downloaded.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(12)
H, W = 96, 96
y, x = np.mgrid[:H, :W]
image = 0.08 + 0.12 * x/W
for cy, cx, ry, rx in [(28, 28, 11, 9), (60, 65, 15, 11), (68, 27, 8, 12)]:
    image += 0.8 * np.exp(-0.5 * (((y-cy)/ry)**2 + ((x-cx)/rx)**2))
image += rng.normal(0, 0.04, (H, W))
image = np.clip(image, 0, 1).astype(np.float32)
print('Image shape:', image.shape, '| dtype:', image.dtype)
print('Top-left pixel:', float(image[0, 0]))
assert image.shape == (96, 96)

## 2 · Show an image and an intensity histogram
`imshow` maps numbers to visual brightness. The histogram shows how often each intensity occurs; it does not tell us where pixels are located.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Simulated fluorescence')
axes[1].hist(image.ravel(), bins=35)
axes[1].set_title('Distribution of pixel values')
axes[1].set_xlabel('Intensity (arbitrary units)')
plt.tight_layout(); plt.show()

## 3 · Array indexing and cropping
The first index selects **row / y**; the second selects **column / x**. A crop uses `[row_start:row_end, column_start:column_end]` with the end excluded.

In [ ]:
crop = image[15:45, 13:48]
print('Crop shape:', crop.shape)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Full frame')
axes[1].imshow(crop, cmap='gray'); axes[1].set_title('Nucleus crop')
plt.tight_layout(); plt.show()
assert crop.shape == (30, 35)

## 4 · Rescale intensity, carefully
A common display transformation maps the minimum value to 0 and the maximum to 1. This changes contrast without changing which pixel belongs to which nucleus. It is **not** a substitute for calibrated intensities when comparing biological samples.

In [ ]:
image_min, image_max = image.min(), image.max()
normalized = (image-image_min) / max(image_max-image_min, 1e-8)
print('Min/max before:', round(float(image_min), 3), round(float(image_max), 3))
print('Min/max after:', normalized.min(), normalized.max())
assert 0 <= normalized.min() <= normalized.max() <= 1

## 5 · A simple threshold mask
A mask is an array of the same width and height, with `True` for foreground pixels and `False` for background. This is a crude first segmentation; gradients and noise complicate the result.

In [ ]:
mask = image > 0.48
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Intensity')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Pixels > 0.48')
plt.tight_layout(); plt.show()
print('Foreground fraction:', round(float(mask.mean()), 3))
assert mask.dtype == bool

## Your turn (10 minutes)
1. Change the threshold from 0.48 to 0.3. What happens to the mask?
2. Crop the nucleus near `(row=60, column=65)` with a 30×30 crop.
3. Change the random seed and see how noise affects the segmentation.

**Self-check:** Is the foreground fraction larger at a *lower* threshold?

In [ ]:
low_threshold_mask = image > 0.30
assert low_threshold_mask.sum() >= mask.sum()
second_crop = image[45:75, 50:80]
assert second_crop.shape == (30, 30)
print('Self-checks passed.')

## Reflect and connect
- Pixel intensity and physical cell size are different quantities.
- Microscopy intensity can depend on exposure time, staining, background and detector settings.
- Next: what counts as *ground truth*, and how do we score a segmentation?